In [ ]:
import sys
import os

# Adds the parent directory to the python path
sys.path.append(os.path.abspath(os.path.join('..')))

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt


from datasets import load_dataset
from torch.utils.data import DataLoader


from src.data.vocabulary import Vocabulary
from src.data.dataset import IMDBDataset
from src.data.dataloader import create_dataloaders

from src.models.bilstm import BiLSTMClassifier
from src.training.trainer import Trainer

from src.evaluation.metrics import (
    calculate_metrics,
    calculate_confusion_matrix,
)

In [ ]:
## CONFIGURATION CELL

MAX_VOCAB_SIZE = 10_000
MAX_LENGTH = 500

EMBEDDING_DIM = 128
HIDDEN_DIM = 128
NUM_LAYERS = 1
DROPOUT = 0.0

BATCH_SIZE = 32
LEARNING_RATE = 1e-3
EPOCHS = 5

SEED = 42

In [ ]:
## reproducibility

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

In [ ]:
## load and prepare the dataset

dataset = load_dataset("stanfordnlp/imdb")

dataset

In [ ]:
# Convert the Hugging Face splits into Python lists.

train_texts_all = dataset["train"]["text"]
train_labels_all = dataset["train"]["label"]

test_texts = dataset["test"]["text"]
test_labels = dataset["test"]["label"]

print(type(train_texts_all))
print(len(train_texts_all))
print(train_labels_all[:10])

In [ ]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_texts_all,
    train_labels_all,
    test_size=0.20,
    random_state=SEED,
    stratify=train_labels_all,
)

print("Training samples  :", len(train_texts))
print("Validation samples:", len(val_texts))
print("Test samples      :", len(test_texts))

In [ ]:
## building the vocabulary

MAX_VOCAB_SIZE = 10_000
MAX_LENGTH = 500

vocab = Vocabulary(max_size=MAX_VOCAB_SIZE)

print("Before build:", len(vocab.word_to_id))

vocab.build(train_texts)

print("After build:", len(vocab.word_to_id))
print("First 20 words:")
print(list(vocab.word_to_id.items())[:20])

In [ ]:
# Vocabulary sanity checks

test_words = [
    "the",
    "and",
    "a",
    "i",
    "have",
    "movie",
    "bond",
    "james",
]

for word in test_words:
    print(f"{word:10s} -> {vocab.word_to_id.get(word)}")

In [ ]:
## pytorch datasets

train_dataset = IMDBDataset(
    texts=train_texts,
    labels=train_labels,
    vocabulary=vocab,
    max_length=MAX_LENGTH,
)

val_dataset = IMDBDataset(
    texts=val_texts,
    labels=val_labels,
    vocabulary=vocab,
    max_length=MAX_LENGTH,
)

test_dataset = IMDBDataset(
    texts=test_texts,
    labels=test_labels,
    vocabulary=vocab,
    max_length=MAX_LENGTH,
)

print("Train dataset:", len(train_dataset))
print("Val dataset  :", len(val_dataset))
print("Test dataset :", len(test_dataset))

In [ ]:
## data loaders

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))
print("Test batches :", len(test_loader))

In [ ]:
## data pipeline sanity check

batch_x, batch_y, batch_lengths = next(iter(train_loader))

print("Input :", batch_x.shape)
print("Labels:", batch_y.shape)
print("Lengths:", batch_lengths.shape)

print("Input dtype :", batch_x.dtype)
print("Label dtype :", batch_y.dtype)

print("Sample lengths:", batch_lengths[:10])
print("Sample labels :", batch_y[:10])

In [ ]:
# Verify padding and sequence lengths.

for i in range(5):
    length = batch_lengths[i].item()
    sequence = batch_x[i]

    real_tokens = (sequence != vocab.word_to_id["<PAD>"]).sum().item()
    padding_tokens = (sequence == vocab.word_to_id["<PAD>"]).sum().item()

    print(
        f"Sample {i}: "
        f"length={length}, "
        f"real={real_tokens}, "
        f"padding={padding_tokens}"
    )

In [ ]:
## BiLSTM model

bilstm_model = BiLSTMClassifier(
    vocab_size=len(vocab.word_to_id),
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
).to(device)

print(bilstm_model)

In [ ]:
# Parameter count

total_parameters = sum(
    parameter.numel()
    for parameter in bilstm_model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in bilstm_model.parameters()
    if parameter.requires_grad
)

print("Total parameters    :", total_parameters)
print("Trainable parameters:", trainable_parameters)

In [ ]:
## forward pass sanity check

batch_x = batch_x.to(device)
batch_lengths = batch_lengths.to(device)

with torch.no_grad():
    output = bilstm_model(
        batch_x,
        batch_lengths,
    )

print("Input :", batch_x.shape)
print("Output:", output.shape)

In [ ]:
## forward pass smoke test

bilstm_model.eval()

with torch.no_grad():
    input_ids, labels, lengths = next(iter(train_loader))

    input_ids = input_ids.to(device)
    lengths = lengths.to(device)

    logits = bilstm_model(
        input_ids,
        lengths,
    )

print("Input :", input_ids.shape)
print("Lengths:", lengths.shape)
print("Output:", logits.shape)

In [ ]:
criterion = nn.BCEWithLogitsLoss()

bilstm_optimizer = torch.optim.Adam(
    bilstm_model.parameters(),
    lr=LEARNING_RATE,
)

In [ ]:
bilstm_trainer = Trainer(
    model=bilstm_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=bilstm_optimizer,
    criterion=criterion,
    device=device,
    checkpoint_path="checkpoints/bilstm/best_model.pt",
)

In [ ]:
# ## single epoch smoke test

# history = bilstm_trainer.fit(epochs=1)

In [ ]:
## run full 5 epochs
history = bilstm_trainer.fit(
    epochs=EPOCHS
)

In [ ]:
## Learning curves

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.plot(
    history["train_loss"],
    label="Train Loss",
)

plt.plot(
    history["val_loss"],
    label="Validation Loss",
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("BiLSTM Training and Validation Loss")
plt.legend()

save_dir = "plots/bilstm/"
os.makedirs(save_dir, exist_ok=True)
plt.savefig(os.path.join(save_dir, "bilstm_loss_curve.png"), dpi=300, bbox_inches="tight")


plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history["train_accuracy"],
    label="Train Accuracy",
)

plt.plot(
    history["val_accuracy"],
    label="Validation Accuracy",
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("BiLSTM Training and Validation Accuracy")
plt.legend()

save_dir = "plots/bilstm/"
os.makedirs(save_dir, exist_ok=True)
plt.savefig(os.path.join(save_dir, "bilstm_accuracy_curve.png"), dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
## loading the best plot

checkpoint_path = "checkpoints/bilstm/best_model.pt"

bilstm_model.load_state_dict(
    torch.load(
        checkpoint_path,
        map_location=device,
    )
)

bilstm_model.eval()
print("Loaded:", checkpoint_path)

In [ ]:
## Validation evaluation

val_loss, val_accuracy = bilstm_trainer.validate()

print("Validation Loss    :", val_loss)
print("Validation Accuracy:", val_accuracy)

In [ ]:
y_true, y_pred, y_prob = bilstm_trainer.predict(
    val_loader
)

validation_metrics = calculate_metrics(
    y_true,
    y_pred,
    y_prob,
)

print(validation_metrics)

In [ ]:
## confusion matrix

cm = calculate_confusion_matrix(
    y_true,
    y_pred,
)

print(cm)

#### Same thing again

In [ ]:
## test evaluation

test_labels, test_predictions, test_probabilities = (
    bilstm_trainer.predict(test_loader)
)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

In [ ]:
bilstm_metrics = {
    "accuracy": accuracy_score(
        test_labels,
        test_predictions,
    ),

    "precision": precision_score(
        test_labels,
        test_predictions,
    ),

    "recall": recall_score(
        test_labels,
        test_predictions,
    ),

    "f1": f1_score(
        test_labels,
        test_predictions,
    ),

    "roc_auc": roc_auc_score(
        test_labels,
        test_probabilities,
    ),
}

print(bilstm_metrics)

In [ ]:
cm = confusion_matrix(
    test_labels,
    test_predictions,
)

print(cm)

In [ ]:
plt.figure(figsize=(6, 5))

plt.imshow(cm)

plt.title("BiLSTM Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")

plt.xticks(
    [0, 1],
    ["Negative", "Positive"],
)

plt.yticks(
    [0, 1],
    ["Negative", "Positive"],
)

plt.colorbar()

for i in range(2):
    for j in range(2):
        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center",
        )

save_dir = "plots/bilstm/"
os.makedirs(save_dir, exist_ok=True)
plt.savefig(os.path.join(save_dir, "bilstm_confusion_matrix.png"), dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
## Final test set evaluation

test_y_true, test_y_pred, test_y_prob = bilstm_trainer.predict(
    test_loader
)

test_metrics = calculate_metrics(
    test_y_true,
    test_y_pred,
    test_y_prob,
)

print("Test Metrics")
print("-" * 40)

for metric_name, value in test_metrics.items():
    print(f"{metric_name:12s}: {float(value):.4f}")

## Experiment Conclusions

The BiLSTM classifier achieved a test accuracy of 85.42%, an F1-score
of 85.22%, and a ROC-AUC of 93.24%.

Compared with the Simple RNN and unidirectional LSTM baselines, the
BiLSTM achieved the strongest overall performance so far.

The BiLSTM achieved higher precision than the LSTM (86.41% vs 81.91%)
while maintaining a strong recall of 84.06%. Its ROC-AUC of 93.24%
also indicates improved overall discrimination between positive and
negative reviews.

The best validation loss was achieved at Epoch 4. Although training
accuracy continued increasing in Epoch 5, validation loss increased,
indicating the beginning of overfitting.

The best validation checkpoint was therefore used for final test
evaluation.